# IntelliDoc AI — Notebook 2: LayoutLMv3 Pretrained Baseline

**Purpose:** Run the pretrained LayoutLMv3 model on the FUNSD test set **without any training** and record the baseline accuracy scores.

**Research meaning:** These scores (Precision, Recall, F1) represent how good the model is before we do anything. After fine-tuning (Notebook 3), we compare to see how much we improved.

---
> **Prerequisites:** Run Notebook 1 first to confirm your FUNSD dataset is correctly placed.

## Step 1: Mount Google Drive and Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
FUNSD_ROOT = '/content/drive/MyDrive/funsd'
TEST_DIR   = os.path.join(FUNSD_ROOT, 'testing_data')

if not os.path.exists(TEST_DIR):
    raise FileNotFoundError('FUNSD test data not found. Please check your Google Drive path.')
print('Paths verified.')

## Step 2: Install Required Libraries
This installs HuggingFace Transformers which gives us access to LayoutLMv3.

In [ ]:
!pip install -q transformers datasets seqeval Pillow timm
print('All libraries installed.')

## Step 3: Load and Preprocess the FUNSD Test Set
We convert the FUNSD annotation JSONs into the format LayoutLMv3 expects:
- A list of words
- The bounding box of each word (normalized to 0–1000 range)
- The label of each word

In [ ]:
import json
import glob
from PIL import Image

# FUNSD label map — maps label names to integer IDs
LABEL2ID = {'other': 0, 'question': 1, 'answer': 2, 'header': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def normalize_bbox(bbox, width, height):
    """Normalize bounding box coordinates to 0-1000 range (LayoutLM format)."""
    x0, y0, x1, y1 = bbox
    return [
        int(1000 * x0 / width),
        int(1000 * y0 / height),
        int(1000 * x1 / width),
        int(1000 * y1 / height),
    ]

def load_funsd_for_layoutlm(split_dir):
    """Load FUNSD annotations and return structured data for LayoutLMv3."""
    ann_dir = os.path.join(split_dir, 'annotations')
    img_dir = os.path.join(split_dir, 'images')
    samples = []

    for ann_file in sorted(glob.glob(os.path.join(ann_dir, '*.json'))):
        fname = os.path.basename(ann_file).replace('.json', '')
        img_path = os.path.join(img_dir, fname + '.png')

        if not os.path.exists(img_path):
            continue

        image = Image.open(img_path).convert('RGB')
        width, height = image.size

        with open(ann_file) as f:
            ann = json.load(f)

        words, bboxes, labels = [], [], []
        for item in ann.get('form', []):
            label_id = LABEL2ID.get(item.get('label', 'other'), 0)
            for word_info in item.get('words', []):
                word = word_info.get('text', '').strip()
                box  = word_info.get('box', [0, 0, 0, 0])
                if word:
                    words.append(word)
                    bboxes.append(normalize_bbox(box, width, height))
                    labels.append(label_id)

        samples.append({'image': image, 'words': words,
                         'bboxes': bboxes, 'labels': labels, 'filename': fname})

    return samples

print('Loading test set...')
test_samples = load_funsd_for_layoutlm(TEST_DIR)
print(f'Loaded {len(test_samples)} test documents.')

## Step 4: Load Pretrained LayoutLMv3
We load `microsoft/layoutlmv3-base` — a model already trained on millions of documents by Microsoft.
**No training happens here.** We just load it and run it.

In [ ]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
import torch

MODEL_NAME = 'microsoft/layoutlmv3-base'
NUM_LABELS = len(LABEL2ID)

print(f'Loading processor from {MODEL_NAME}...')
processor = LayoutLMv3Processor.from_pretrained(MODEL_NAME, apply_ocr=False)

print(f'Loading model from {MODEL_NAME}...')
model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True   # classification head is randomly initialized
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
print(f'Model ready on device: {device}')

## Step 5: Run Inference on All Test Documents
Feed each test document through the model and collect the predicted labels.

In [ ]:
from tqdm.auto import tqdm

all_true_labels = []
all_pred_labels = []

print('Running inference on test documents...')
for sample in tqdm(test_samples):
    words   = sample['words']
    bboxes  = sample['bboxes']
    labels  = sample['labels']
    image   = sample['image']

    if not words:
        continue

    encoding = processor(
        image,
        words,
        boxes=bboxes,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors='pt'
    )

    # Move inputs to GPU if available
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**encoding)

    logits     = outputs.logits[0]  # shape: (seq_len, num_labels)
    predictions = logits.argmax(dim=-1).cpu().tolist()

    # Map word-piece tokens back to words using word_ids
    word_ids   = encoding.word_ids() if hasattr(encoding, 'word_ids') else None

    # Collect per-word predictions (take first token of each word)
    seen_word_ids = set()
    word_preds = []
    word_trues = []

    if word_ids:
        for idx, wid in enumerate(word_ids):
            if wid is None or wid in seen_word_ids:
                continue
            seen_word_ids.add(wid)
            if wid < len(labels):
                word_preds.append(ID2LABEL[predictions[idx]])
                word_trues.append(ID2LABEL[labels[wid]])
    else:
        # Fallback: align by position
        n = min(len(labels), len(predictions))
        word_preds = [ID2LABEL[predictions[i]] for i in range(n)]
        word_trues = [ID2LABEL[labels[i]] for i in range(n)]

    all_true_labels.append(word_trues)
    all_pred_labels.append(word_preds)

print(f'Inference complete. Evaluated {len(all_true_labels)} documents.')

## Step 6: Compute Baseline Scores
Calculate **Precision, Recall, and F1** for each label and overall.
These are your **Baseline scores** — write them down for your research records.

In [ ]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

# seqeval expects BIO-tagged format, so we wrap our flat labels in lists
report = classification_report(all_true_labels, all_pred_labels, digits=4)

print('=' * 60)
print('INTELLIDOC AI — LAYOUTLMV3 PRETRAINED BASELINE RESULTS')
print('Model  : microsoft/layoutlmv3-base  (NO fine-tuning)')
print('Dataset: FUNSD Test Set')
print('=' * 60)
print(report)
print()
print(f'Overall F1 Score  : {f1_score(all_true_labels, all_pred_labels):.4f}')
print(f'Overall Precision : {precision_score(all_true_labels, all_pred_labels):.4f}')
print(f'Overall Recall    : {recall_score(all_true_labels, all_pred_labels):.4f}')
print('=' * 60)
print()
print('IMPORTANT: Save these numbers. They are your research baseline.')
print('After fine-tuning (Notebook 3), compare new scores to these.')

## Step 7: Visualize One Prediction vs Ground Truth
See which labels the model got right and which it got wrong on a real document.

In [ ]:
import matplotlib.pyplot as plt
from PIL import ImageDraw

COLORS = {'question': 'blue', 'answer': 'green', 'header': 'orange', 'other': 'gray'}

def visualize_predictions(sample, true_labels, pred_labels, max_words=80):
    image  = sample['image'].copy()
    bboxes = sample['bboxes']
    width, height = image.size

    fig, axes = plt.subplots(1, 2, figsize=(18, 12))
    for ax, labels, title in [
        (axes[0], true_labels, 'Ground Truth'),
        (axes[1], pred_labels, 'Model Prediction (Baseline)')
    ]:
        img_copy = image.copy()
        draw = ImageDraw.Draw(img_copy)
        for i, (bbox, label) in enumerate(zip(bboxes[:max_words], labels[:max_words])):
            x0 = int(bbox[0] * width  / 1000)
            y0 = int(bbox[1] * height / 1000)
            x1 = int(bbox[2] * width  / 1000)
            y1 = int(bbox[3] * height / 1000)
            color = COLORS.get(label, 'gray')
            draw.rectangle([x0, y0, x1, y1], outline=color, width=2)
        ax.imshow(img_copy)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')

    plt.suptitle(f"Document: {sample['filename']}", fontsize=12)
    plt.tight_layout()
    plt.show()

# Visualize the first test document
if all_true_labels and all_pred_labels:
    sample_to_show = test_samples[0]
    visualize_predictions(
        sample_to_show,
        all_true_labels[0],
        all_pred_labels[0]
    )

## ✅ Baseline Complete

You now have your **Baseline F1 scores** for LayoutLMv3 on FUNSD.

**Record these numbers in your research notes:**
- Overall F1: ___
- Question F1: ___
- Answer F1: ___
- Header F1: ___

**Next step:** Open `03_layoutlmv3_finetune.ipynb` to fine-tune the model and see how much it improves.